# 08. Agentic RAG — 스스로 판단하는 RAG 만들기

지금까지 세 가지 검색 방법을 만들었습니다.

| 노트북 | 검색 방법 | 잘하는 질문 |
|---|---|---|
| 02 | **SQL**(관계형 DB) | "비저항이 5 μΩ·cm 이하인 금속은?" 같은 **정확한 수치·조건** |
| 06 | **벡터 검색** | "결정립계 편석이 비저항에 어떤 영향을 주나?" 같은 **본문 설명** |
| 07 | **그래프 검색** | "Cu를 대체할 수 있는 금속들은?" 같은 **개체 사이의 관계** |

그런데 06의 벡터 RAG는 질문이 무엇이든 **늘 같은 길**(벡터 검색 1번 → 답변)로만 갑니다.
숙련된 연구원이라면 질문을 보고 "이건 물성표를 찾아보자", "이건 논문 본문을 읽어 보자"라고 **판단**하고,
찾은 자료가 부족하면 **검색어를 바꿔 다시 찾아볼** 것입니다.

**Agentic RAG**는 이 판단을 LLM에게 맡긴 RAG입니다. 05에서 배운 **에이전트**(도구를 쓰는 LLM)와 RAG를 합친 것이죠.

## 학습 목표
1. Agentic RAG의 핵심 단계(**라우팅 → 검색 → 평가 → 질문 재작성 → 답변**)를 이해합니다.
2. 순수 파이썬 `while` 반복문으로 Agentic RAG를 **직접** 만들어 봅니다.
3. 같은 흐름을 **LangGraph**의 `StateGraph`로 옮겨, 흐름도를 그림으로 확인합니다.
4. 검색 도구 3개를 쥐여 준 **도구 호출 에이전트**(05의 `create_agent`)와 비교합니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from skku_rag import LLM_MODEL, EMBED_MODEL, get_llm

llm = get_llm()
print("LLM:", LLM_MODEL, "| 임베딩:", EMBED_MODEL)

## 1. 검색 도구 3개 준비하기

02, 06, 07 노트북에서 **여러분이 직접 만든** 저장소를 불러옵니다. 각 노트북에서 했던 일을 `skku_rag` 도우미 함수로 묶어 두었기 때문에 한 줄씩이면 됩니다.

- SQLite와 Chroma는 `db/` 폴더에 없으면 이 자리에서 새로 만듭니다(1~3분).
- **지식그래프는 07 노트북에서 LLM으로 개체·관계를 추출해야만 만들어집니다.** 07을 아직 실행하지 않았다면 먼저 실행하세요. (오류 메시지가 안내해 줍니다.)

| 저장소 | 불러오는 함수 | 만든 노트북 |
|---|---|---|
| SQLite (물성표) | `build_sqlite()` | 02 |
| Chroma (논문 조각 벡터) | `get_vectorstore()` | 06 |
| Kùzu (지식그래프) | `get_graph()` | 07 |

> **잠금 오류가 나면?** `Could not set lock on file ... paper_graph.kuzu` 오류는 다른 노트북(주로 07)이 같은 그래프 DB 파일을 열어 두고 있다는 뜻입니다.
> Kùzu 같은 내장형 DB는 **한 번에 한 프로그램만** 파일을 쓸 수 있기 때문입니다. 07 노트북의 메뉴에서 `Kernel → Shut Down Kernel`(Colab: `런타임 → 세션 종료`)을 누른 뒤 다시 실행하세요.

In [ ]:
import sqlite3
from skku_rag.sql_store import build_sqlite, SQL_SCHEMA_DESCRIPTION
from skku_rag.vector_store import get_vectorstore, format_docs
from skku_rag.graph_store import get_graph, find_entities, get_neighbors

sql_path = build_sqlite()          # 02: 물성표 SQLite
vectorstore = get_vectorstore()    # 06: 논문 조각 벡터DB
graph = get_graph()                # 07: 논문 지식그래프

print("SQLite :", sql_path.name)
print("Chroma :", vectorstore._collection.count(), "개 조각")
print("Kùzu   :", graph.execute("MATCH (e:Entity) RETURN count(e)").get_next()[0], "개 개체")

### 1-1. 도구 ① 논문 본문 검색 (벡터)
06에서 만든 것과 같습니다. 질문과 의미가 가까운 논문 조각 `k`개를 찾아 출처와 함께 돌려줍니다.

In [ ]:
def search_papers(query: str, k: int = 4) -> str:
    """논문 본문에서 질문과 의미가 가까운 조각을 찾아 출처와 함께 돌려줍니다."""
    docs = vectorstore.similarity_search(query, k=k)
    return format_docs(docs)

print(search_papers("grain boundary segregation resistivity", k=1)[:500])

### 1-2. 도구 ② 물성 DB 조회 (Text2SQL)
02에서 배운 Text2SQL입니다. LLM이 질문을 SQL로 바꾸고, 우리는 그 SQL을 **읽기 전용**으로 실행합니다.

- `SELECT`로 시작하지 않는 SQL은 거절합니다. LLM이 실수로 `DELETE` 같은 명령을 만들어도 데이터가 지워지지 않게 하기 위한 안전장치입니다.
- DB도 `mode=ro`(read-only)로 열어 이중으로 막습니다.
- 실행 중 오류가 나면 오류 메시지를 LLM에게 보여 주고 **한 번 더** 고치게 합니다.

In [ ]:
import re
from contextlib import closing
from pathlib import Path

SQL_PROMPT = """You write one SQLite SELECT query that answers the question.
{schema}
Rules: return ONLY the SQL, no explanation, no ``` fences. Use only the tables/columns above.
Keep it simple: one table, no self-joins. For representative metal values use source = 'sung2026 Table 1'.
{error}
Question: {question}
SQL:"""

def _extract_sql(text: str) -> str:
    """LLM 응답에서 SQL 문만 골라냅니다(코드블록 기호 등 제거)."""
    text = re.sub(r"```(?:sql)?", "", text, flags=re.I).strip()
    match = re.search(r"(select|with)\b.*", text, flags=re.I | re.S)
    return match.group(0).split(";")[0].strip() if match else text

def query_metal_db(question: str, max_tries: int = 2) -> str:
    """금속 물성표(SQLite)를 Text2SQL 로 조회해 SQL 과 결과 표를 문자열로 돌려줍니다."""
    error = ""
    for _ in range(max_tries):
        prompt = SQL_PROMPT.format(schema=SQL_SCHEMA_DESCRIPTION, error=error, question=question)
        sql = _extract_sql(llm.invoke(prompt).content)
        if not sql.lower().startswith(("select", "with")):
            error = f"Previous answer was not a SELECT query: {sql[:80]}"
            continue
        try:
            # mode=ro: 읽기 전용으로 열기 / closing: 다 쓰면 연결을 확실히 닫기
            with closing(sqlite3.connect(Path(sql_path).resolve().as_uri() + "?mode=ro", uri=True)) as conn:
                cur = conn.execute(sql)
                header = [c[0] for c in cur.description]
                rows = cur.fetchmany(15)
            table = "\n".join([" | ".join(header)] + [" | ".join(map(str, r)) for r in rows])
            return f"[SQL]\n{sql}\n[결과]\n{table if rows else '(결과 없음)'}"
        except sqlite3.Error as e:
            error = f"Previous SQL `{sql}` failed with error: {e}. Fix it."
    return f"[SQL 실패] {error}"

print(query_metal_db("List the 3 element metals with the lowest bulk resistivity."))

### 1-3. 도구 ③ 지식그래프 탐색
07에서 배운 방법입니다.
1. LLM이 질문에서 핵심 개체(재료, 현상 등)를 **영어 이름**으로 뽑습니다. (그래프에는 영어 이름으로 저장되어 있으니까요.)
2. 그래프에서 그 개체를 찾고(`find_entities`), 연결된 관계를 모읍니다(`get_neighbors`).
3. 관계를 `A -[관계]-> B` 형태의 글로 만들어 돌려줍니다.

In [ ]:
ENTITY_PROMPT = """Extract the key materials, properties or phenomena from the question
as short English names (chemical formula for elements, e.g. Cu, Ru; lowercase words otherwise,
e.g. resistivity, grain boundary scattering). Return them comma-separated, nothing else.
Question: {question}
Names:"""

import threading
graph_lock = threading.Lock()  # 에이전트가 도구를 동시에 부를 때 그래프 DB 연결을 한 번에 하나씩만 쓰도록

def explore_graph(question: str, max_edges: int = 25) -> str:
    """질문 속 개체를 지식그래프에서 찾아, 연결된 관계들을 문장 목록으로 돌려줍니다."""
    raw = llm.invoke(ENTITY_PROMPT.format(question=question)).content
    keywords = [w.strip(" .'\"") for w in raw.split(",") if w.strip(" .'\"")][:5]
    with graph_lock:
        found = find_entities(graph, keywords, limit=5)
        if found.empty:
            return f"(그래프에서 개체를 찾지 못했습니다. 추출한 키워드: {keywords})"
        edges = get_neighbors(graph, list(found.name), hops=1, limit=max_edges)
    lines = [f"{r.source} -[{r.relation}]-> {r.target}" for r in edges.itertuples()]
    return f"[찾은 개체] {', '.join(found.name)}\n[관계]\n" + "\n".join(lines)

print(explore_graph("구리를 대체할 수 있는 금속에는 무엇이 있나요?"))

## 2. Agentic RAG의 흐름

06의 벡터 RAG와 비교해 보면 차이가 분명합니다.

```
[06 벡터 RAG]   질문 ──▶ 벡터 검색 ──▶ 답변                (항상 같은 길, 한 번만)

[08 Agentic RAG]
                 ┌──────────────────────────────────────────────┐
                 ▼                                              │
  질문 ──▶ ① 라우팅 ──▶ ② 검색 ──▶ ③ 평가 ──(충분함)──▶ ⑤ 답변   │
          (어느 도구?)  (SQL/벡터/   (자료가 질문에                 │
                        그래프)      답할 만한가?)                  │
                                       │(부족함)                   │
                                       ▼                          │
                                  ④ 질문 재작성 ──────────────────┘
                                  (검색어를 바꿔 다시)
```

| 단계 | 하는 일 | 실험실 비유 |
|---|---|---|
| ① 라우팅(Routing) | 질문에 맞는 검색 도구 고르기 | "이건 물성 핸드북을 볼까, 논문을 읽을까?" |
| ② 검색(Retrieve) | 고른 도구로 자료 찾기 | 핸드북·논문 찾아보기 |
| ③ 평가(Grade) | 찾은 자료가 질문에 답할 만한지 판단 | "이 자료로 결론을 낼 수 있나?" |
| ④ 재작성(Rewrite) | 검색어를 더 좋게 바꾸기 | "키워드를 바꿔서 다시 찾아보자" |
| ⑤ 답변(Generate) | 자료를 근거로 답 쓰기 | 보고서 작성 |

각 단계는 **작은 LLM 호출 하나**입니다. 이제 하나씩 만들어 봅니다.

## 3. [나이브 구현] 순수 파이썬으로 만들기

### 3-1. ① 라우터 — 어느 도구를 쓸까?
LLM에게 도구 설명을 보여 주고 `sql`, `vector`, `graph` 중 **한 단어**로만 답하게 합니다.
소형 모델은 가끔 형식을 어기므로, 답에서 세 단어 중 하나를 찾아내고 못 찾으면 가장 무난한 `vector`를 씁니다.

In [ ]:
ROUTER_PROMPT = """You route a materials-science question to ONE data source.
- sql: numeric property table of metals (resistivity, electron mean free path, rho*lambda, cohesive energy)
       and Cu alloy films (solute at.%, grain size). Use for numbers, rankings, comparisons, filters.
- graph: knowledge graph of entities and relations (which material is an alternative to which,
         what increases/decreases resistivity, what contains what). Use for "which/what are related" questions.
- vector: full text of two papers. Use for explanations, mechanisms, reasons, experimental details.
Answer with exactly one word: sql, graph, or vector.
Question: {question}
Answer:"""

def route(question: str) -> str:
    """질문을 보고 'sql' / 'graph' / 'vector' 중 하나를 고릅니다."""
    answer = llm.invoke(ROUTER_PROMPT.format(question=question)).content.lower()
    for source in ("sql", "graph", "vector"):
        if source in answer:
            return source
    return "vector"  # 형식을 어기면 가장 무난한 벡터 검색으로

for q in ["Ru와 Co 중 벌크 비저항이 더 낮은 금속은?",
          "구리를 대체할 수 있는 배선 금속에는 무엇이 있나요?",
          "Cu-Nb 합금에서 결정립이 오히려 커지는 이유는?"]:
    print(f"{route(q):6s} ← {q}")

### 3-2. ② 검색 — 고른 도구 실행하기
도구 이름과 함수를 딕셔너리로 묶어 두면, 라우터가 고른 이름으로 바로 함수를 꺼내 쓸 수 있습니다.

In [ ]:
TOOLS = {"sql": query_metal_db, "vector": search_papers, "graph": explore_graph}

def retrieve(source: str, query: str) -> str:
    return TOOLS[source](query)

### 3-3. ③ 평가 — 찾은 자료가 쓸 만한가?
찾은 자료(context)와 질문을 보여 주고 `yes` / `no`로 판정하게 합니다.
이 단계 덕분에 엉뚱한 자료로 그럴듯한 답을 지어내는(환각) 일을 줄일 수 있습니다.

In [ ]:
GRADE_PROMPT = """Does the context contain information that helps answer the question?
Answer only "yes" or "no".
Question: {question}
Context:
{context}
Answer:"""

def grade(question: str, context: str) -> bool:
    answer = llm.invoke(GRADE_PROMPT.format(question=question, context=context[:3000])).content
    return "yes" in answer.lower()

### 3-4. ④ 질문 재작성 — 검색어 바꾸기
학생 질문은 한국어인데 논문과 그래프는 영어입니다. 자료가 부족하다고 판정되면,
질문을 **논문에 나올 법한 영어 검색어**로 바꾸고, 필요하면 **다른 도구**도 골라 다시 시도합니다.

In [ ]:
REWRITE_PROMPT = """The search below did not find useful information.
Rewrite the question as a short, specific English search query that would match
wording in materials-science papers (use technical terms and chemical formulas).
Return only the new query.
Original question: {question}
Previous query: {query}
New query:"""

def rewrite(question: str, query: str) -> str:
    return llm.invoke(REWRITE_PROMPT.format(question=question, query=query)).content.strip().strip('"')

### 3-5. ⑤ 답변 생성
06에서 쓴 RAG 프롬프트와 거의 같습니다. **자료에 없는 내용은 모른다고 답하라**는 지시가 핵심입니다.

In [ ]:
ANSWER_PROMPT = """당신은 소재공학 연구를 돕는 조수입니다. 아래 [자료]만 근거로 질문에 한국어로 답하세요.
자료에 없는 내용은 지어내지 말고 "자료에서 찾을 수 없습니다"라고 답하세요.
수치에는 단위를 붙이고, 가능하면 출처(논문/표/쪽)를 함께 적으세요.

[자료]
{context}

[질문]
{question}

[답변]"""

def generate(question: str, context: str) -> str:
    return llm.invoke(ANSWER_PROMPT.format(question=question, context=context)).content

### 3-6. 모두 합치기 — `while` 반복문 하나면 됩니다
05에서 ReAct 에이전트를 `while`문으로 만들었던 것을 떠올려 보세요. 여기서도 같습니다.

- 평가를 통과하면 반복을 멈추고 답변합니다.
- 통과하지 못하면 질문을 바꾸고 도구를 다시 골라 재시도합니다.
- **최대 시도 횟수**를 정해 두어야 무한 반복에 빠지지 않습니다.

In [ ]:
def agentic_rag(question: str, max_tries: int = 3, verbose: bool = True) -> str:
    log = print if verbose else (lambda *a, **k: None)
    query = question
    for attempt in range(1, max_tries + 1):
        source = route(query)                                   # ① 라우팅
        context = retrieve(source, query)                       # ② 검색
        ok = grade(question, context)                           # ③ 평가
        log(f"[시도 {attempt}] 도구={source:6s} | 검색어={query[:60]!r} | 평가={'통과' if ok else '부족'}")
        if ok:
            break
        query = rewrite(question, query)                        # ④ 재작성
    else:
        log("→ 최대 시도 횟수 도달: 마지막 자료로 답변합니다(자료가 부족할 수 있음).")
    return generate(question, context)                          # ⑤ 답변

### 3-7. 실행해 보기 — 세 종류의 질문

In [ ]:
answer = agentic_rag("벌크 비저항이 가장 낮은 원소 금속 3개와 그 값을 알려 주세요.")
print("\n[답변]\n" + answer)

In [ ]:
answer = agentic_rag("구리(Cu)를 대체할 수 있는 배선 금속 후보에는 무엇이 있나요?")
print("\n[답변]\n" + answer)

In [ ]:
answer = agentic_rag("Cu-Nb 합금에서 Nb를 조금 넣었을 때 결정립 크기와 비저항은 어떻게 변하나요?")
print("\n[답변]\n" + answer)

> **결과 읽는 법**
> - 줄마다 `[시도 n]` 로그가 찍힙니다. 라우터가 어떤 도구를 골랐는지, 평가를 통과했는지 확인해 보세요.
> - 소형 모델(4B)이라 라우팅이 기대와 다르거나(예: 수치 질문에 `vector` 선택) 평가가 너그러울 때가 있습니다.
>   이것이 바로 "LLM에게 판단을 맡길 때의 위험"입니다. 더 큰 모델로 바꾸면 판단이 정확해집니다(00 노트북 참고).

## 4. [LangGraph] 같은 흐름을 그래프로 만들기

3장에서 만든 `while` 반복문은 짧지만, 단계가 늘어나면(예: 답변 검증, 웹 검색 추가) 흐름이 복잡해집니다.
**LangGraph**는 이런 흐름을 **노드(할 일)**와 **엣지(다음에 갈 곳)**로 그리는 도구입니다.
04에서 배운 그래프 개념이 여기서도 쓰입니다!

| LangGraph 용어 | 뜻 | 여기서는 |
|---|---|---|
| State(상태) | 단계 사이에 들고 다니는 메모장 | 질문, 검색어, 도구, 자료, 시도 횟수, 답변 |
| Node(노드) | 한 단계의 일을 하는 함수 | route, retrieve, rewrite, generate |
| Edge(엣지) | 다음 노드로 가는 화살표 | route → retrieve |
| Conditional Edge | 조건에 따라 갈림길 | 평가 결과에 따라 generate 또는 rewrite |

노드 함수는 **상태를 받아, 바뀐 값만 딕셔너리로 돌려줍니다.** 나머지는 LangGraph가 상태에 합쳐 줍니다.
3장에서 만든 함수들을 그대로 재사용합니다.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    question: str   # 원래 질문
    query: str      # 현재 검색어 (재작성되면 바뀜)
    source: str     # 고른 도구
    context: str    # 검색된 자료
    tries: int      # 지금까지 검색한 횟수
    answer: str     # 최종 답변

def route_node(state: RAGState) -> dict:
    return {"source": route(state["query"])}

def retrieve_node(state: RAGState) -> dict:
    return {"context": retrieve(state["source"], state["query"]), "tries": state["tries"] + 1}

def rewrite_node(state: RAGState) -> dict:
    return {"query": rewrite(state["question"], state["query"])}

def generate_node(state: RAGState) -> dict:
    return {"answer": generate(state["question"], state["context"])}

평가(grade)는 노드가 아니라 **갈림길을 정하는 함수**로 씁니다. 다음에 갈 노드의 이름을 돌려주면 됩니다.

In [ ]:
MAX_TRIES = 3

def decide_after_retrieve(state: RAGState) -> str:
    """평가를 통과했거나 시도 횟수를 다 쓰면 답변, 아니면 재작성으로 갑니다."""
    if state["tries"] >= MAX_TRIES or grade(state["question"], state["context"]):
        return "generate"
    return "rewrite"

workflow = StateGraph(RAGState)
workflow.add_node("route", route_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("rewrite", rewrite_node)
workflow.add_node("generate", generate_node)

workflow.add_edge(START, "route")
workflow.add_edge("route", "retrieve")
workflow.add_conditional_edges("retrieve", decide_after_retrieve, ["generate", "rewrite"])
workflow.add_edge("rewrite", "route")   # 검색어를 바꾸면 도구도 다시 고름
workflow.add_edge("generate", END)

app = workflow.compile()

만든 흐름을 그림으로 확인해 봅니다. `draw_mermaid()`는 흐름도를 **Mermaid** 형식의 글로 출력합니다.
출력된 글을 [mermaid.live](https://mermaid.live)에 붙여 넣으면 그림으로 볼 수 있습니다.

In [ ]:
print(app.get_graph().draw_mermaid())

위 Mermaid 글을 그림으로 옮기면 다음과 같습니다.

```
            START
              │
              ▼
        ┌──▶ route ───▶ retrieve ──(평가 통과 또는 3회)──▶ generate ──▶ END
        │                  │
        │             (자료 부족)
        │                  ▼
        └────────────── rewrite
```

이제 실행합니다. `stream(..., stream_mode="updates")`를 쓰면 **노드가 하나 끝날 때마다** 무엇이 바뀌었는지 볼 수 있습니다.

In [ ]:
def run_graph(question: str) -> str:
    init = {"question": question, "query": question, "source": "", "context": "", "tries": 0, "answer": ""}
    final = {}
    for update in app.stream(init, stream_mode="updates"):
        for node, changed in update.items():
            final.update(changed)
            summary = {k: (v[:60] + "…" if isinstance(v, str) and len(v) > 60 else v) for k, v in changed.items()}
            print(f"[{node:8s}] {summary}")
    return final.get("answer", "")

answer = run_graph("Ru의 전자 평균 자유 행로(EMFP)는 Cu와 비교해 어떤가요?")
print("\n[답변]\n" + answer)

3장의 `while`문과 **같은 일**을 했다는 것을 확인해 보세요. LangGraph는 마법이 아니라,
우리가 직접 짠 흐름을 "노드와 화살표"로 정리해 주는 틀입니다. 덕분에

- 흐름도를 그림으로 그려 확인할 수 있고,
- 새 단계(예: 답변이 자료와 맞는지 검사하는 노드)를 **노드 하나 + 엣지 하나**로 끼워 넣기 쉽습니다.

## 5. [다른 방식] 도구 호출 에이전트에게 맡기기

3·4장은 **우리가 흐름(라우팅→검색→평가→…)을 정해 주고**, 각 단계의 판단만 LLM에게 맡겼습니다.
05에서 만든 **도구 호출 에이전트**는 더 자유롭습니다. 검색 도구 3개를 쥐여 주면 LLM이 직접

- 어떤 도구를 **몇 번, 어떤 순서로** 쓸지 정하고,
- 필요하면 **여러 도구의 결과를 합쳐** 답합니다.

05에서처럼 `@tool`로 도구를 만들고 `create_agent`에 넘깁니다. **docstring(설명서)이 LLM이 도구를 고르는 유일한 단서**라는 점을 기억하세요.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def paper_search(query: str) -> str:
    """Search the full text of two materials-science papers (Cu interconnects, Cu-Nb/Cu-Fe alloys,
    alternative metals). Use for explanations, mechanisms and experimental details. Query in English."""
    return search_papers(query)

@tool
def metal_property_db(question: str) -> str:
    """Look up numeric properties of metals (bulk resistivity, electron mean free path, rho*lambda,
    cohesive energy) and Cu alloy films (solute at.%, grain size). Use for numbers and rankings."""
    return query_metal_db(question)

@tool
def knowledge_graph(question: str) -> str:
    """Explore relations between materials, properties and phenomena
    (e.g. alternatives to Cu, what increases resistivity)."""
    return explore_graph(question)

agent = create_agent(
    model=llm,
    tools=[paper_search, metal_property_db, knowledge_graph],
    system_prompt=("You are a research assistant for materials scientists. Use the tools to find facts, "
                   "then answer in Korean with units and sources. If the tools give nothing, say so."),
)

In [ ]:
def show_agent(question: str) -> None:
    """에이전트의 도구 주문과 최종 답을 단계별로 출력합니다."""
    inputs = {"messages": [{"role": "user", "content": question}]}
    for update in agent.stream(inputs, stream_mode="updates"):
        for node, data in update.items():
            for m in data["messages"]:
                if getattr(m, "tool_calls", None):
                    print(f"[{node}] 도구 주문:", [(c["name"], c["args"]) for c in m.tool_calls])
                elif node == "tools":
                    print(f"[{node}] 결과 {len(str(m.content))}자 받음")
                else:
                    print(f"[{node}] 최종 답변:\n{m.content}")

show_agent("Ru의 벌크 비저항 수치를 알려 주고, 논문에서 Ru가 배선 재료로 주목받는 이유를 설명해 주세요.")

> 두 가지 정보(수치 + 설명)가 필요한 질문이라, 에이전트가 **도구를 두 번 이상** 부르는지 확인해 보세요.
> 소형 모델은 도구를 한 번만 부르고 끝내거나, 도구 이름·인자를 잘못 줄 때도 있습니다.

## 6. 두 방식 비교와 정리

| | 정해진 흐름 (3·4장, "워크플로우") | 도구 호출 에이전트 (5장) |
|---|---|---|
| 흐름을 정하는 주체 | **사람**이 설계 (라우팅→검색→평가→…) | **LLM**이 스스로 |
| 예측 가능성 | 높음 — 매번 비슷한 경로 | 낮음 — 질문마다 경로가 달라짐 |
| 유연성 | 설계한 경로만 가능 | 여러 도구 조합 가능 |
| 소형 모델과의 궁합 | **좋음** (각 단계가 쉬운 판단) | 불안정할 수 있음 |
| 디버깅 | 쉬움 (단계별 로그) | 어려움 |

그리고 지금까지의 RAG를 한눈에 비교하면:

| | 06 Vector RAG | 07 Graph RAG | 08 Agentic RAG |
|---|---|---|---|
| 검색 대상 | 논문 조각 | 개체·관계 그래프 | **SQL + 벡터 + 그래프 중 선택** |
| 검색 횟수 | 1번 | 1번 | 필요한 만큼 (최대 횟수 제한) |
| 자료 품질 확인 | 없음 | 없음 | **평가 단계** |
| 비용(LLM 호출 수) | 적음 (1회) | 보통 | 많음 (단계마다 호출) |
| 잘 맞는 상황 | 본문 설명 질문 | 관계 질문 | **질문 종류가 섞여 있을 때** |

### 정리
- **Agentic RAG = RAG + 에이전트의 판단.** 어떤 자료를 볼지, 자료가 충분한지, 다시 찾을지를 LLM이 정합니다.
- 핵심 부품은 모두 **짧은 프롬프트 하나짜리 LLM 호출**입니다: 라우터, 평가자, 재작성기, 답변 생성기.
- 순수 파이썬 `while`문으로 만든 것과 LangGraph `StateGraph`로 만든 것은 **같은 일**을 합니다. LangGraph는 흐름을 정리하고 확장하기 쉽게 해 줄 뿐입니다.
- 판단을 LLM에게 맡길수록 **모델 성능**이 결과를 크게 좌우합니다. 실제 연구에 쓸 때는 더 큰 모델과 단계별 로그 확인이 필수입니다.

## 직접 해보기
1. `ROUTER_PROMPT`에 도구 설명을 바꾸거나 예시 질문을 추가해, 3-1의 라우팅 결과가 어떻게 달라지는지 확인해 보세요.
2. 논문에 없는 질문(예: "그래핀의 밴드갭은?")을 `agentic_rag`에 넣어 보세요. 평가 단계가 몇 번 '부족'을 내고, 최종 답변은 어떻게 나오나요?
3. (도전) 4장의 LangGraph에 **답변 검증 노드**를 추가해 보세요. 답변이 자료에 근거했는지 LLM에게 `yes/no`로 묻고, `no`면 다시 `generate`로 보내는 조건부 엣지를 만들면 됩니다.

## 참고자료
- 테디노트, [LangGraph Agentic RAG](https://github.com/teddylee777/langchain-kr/tree/main/17-LangGraph/02-Structures) (Naive RAG → Query Rewrite → Agentic RAG → Adaptive RAG)
- 공원나연, [Neo4j GraphRAG 파이썬 패키지 가이드북 — 랭그래프 기반 Text2Cypher GraphRAG Agent](https://wikidocs.net/book/16760)
- [LangGraph 공식 문서](https://langchain-ai.github.io/langgraph/)